# 03. Análisis de KPIs de PlayLoop

In [12]:
# Importamos librerias
import pandas as pd
from pathlib import Path
from dotenv import dotenv_values
from sqlalchemy import URL, create_engine, text

In [13]:
# Localizamos la raiz
# Esto funciona si el notebook se ejecuta desde la raíz o desde notebooks.
carpeta_actual = Path.cwd()
raiz_proyecto = (
    carpeta_actual.parent
    if carpeta_actual.name == 'notebooks'
    else carpeta_actual
)

# Leemos credenciales sin mostrarlas 
configuracion =  dotenv_values(raiz_proyecto / '.env')

# Preparamos la dirección de conexión a PostgreSQL.
url = URL.create(
    'postgresql+psycopg2',
    username=configuracion['DB_USER'],
    password=configuracion['DB_PASSWORD'],
    host=configuracion['DB_HOST'],
    port=int(configuracion['DB_PORT']),
    database=configuracion['DB_NAME']
)

# Creamos el gestor de conexiones.
engine = create_engine(url, connect_args={'connect_timeout': 5})

In [14]:
# Contamos los usuarios almacenados en PostgreSQL.
consulta = text("""
    SELECT COUNT(*) AS total_usuarios
    FROM public.usuarios
""")

# Ejecutamos la consulta y guardamos el resultado en un DataFrame.
with engine.connect() as conexion:
    resultado = pd.read_sql(consulta, conexion)

# Mostramos la tabla resultante en el notebook.
resultado

,total_usuarios
0,8000


### Parte 1: Retención D1/D7/D30, DAU/MAU y stickiness, y el embudo de niveles 
**Objetivo:** analizar si los jugadores vuelven, con qué frecuencia juegan y hasta dónde avanzan en los primeros niveles.

#### DAU - Usuarios activos diarios
¿Cuántos jugadores distintos abren el juego cada día?
Consideramos activo a un jugador si tiene al menos una sesión ese día.
- **DAU medio**: promedio de jugadores únicos por día sobre todos los días de actividad

In [15]:
# Calculamos los usuarios únicos de cada día y después su promedio.
consulta_dau = text("""
    -- Creamos un resultado temporal con una fila por fecha.
    WITH activos_por_dia AS (
        SELECT
            fecha,
            COUNT(DISTINCT user_id) AS activos
        FROM public.sesiones
        GROUP BY fecha
    )

    -- Calculamos el promedio de los recuentos diarios.
    SELECT
        ROUND(AVG(activos), 0) AS dau_medio,
        COUNT(*) AS dias_con_actividad
    FROM activos_por_dia
""")

# Ejecutamos la consulta y traemos el resultado a pandas.
with engine.connect() as conexion:
    resumen_dau = pd.read_sql(consulta_dau, conexion)

resumen_dau

,dau_medio,dias_con_actividad
0,601.0,56


**Resumen — DAU medio**
- **Resultado:** 601 jugadores diarios de media (601,48 sin redondear), en 56 días.
- **Periodo:** 28/05/2024–22/07/2024. No se añaden ceros para fechas sin actividad.

#### MAU y stickiness 
- **MAU**: jugadores únicos con sesiones en los 30 días previos a la extracción del 22/07/2024
- **Stickiness:** DAU medio / MAU × 100.


In [18]:
consulta_mau = text("""
    WITH mau AS(
-- MAU: jugadores unicos en los 30 dias previos a la extraccion
       SELECT COUNT(DISTINCT user_id) AS mau
        FROM public.sesiones
        WHERE fecha > DATE '2024-07-22' - 30
    ),

    dau AS (
        -- Promedio de usuarios activos de todos los días con actividad.
        SELECT AVG(activos) AS dau
        FROM (
            SELECT
                fecha,
                COUNT(DISTINCT user_id) AS activos
            FROM public.sesiones
            GROUP BY fecha
        ) AS activos_por_dia
    )

    SELECT
        mau.mau,
        ROUND(dau.dau, 0) AS dau_medio,
        ROUND(100.0 * dau.dau / NULLIF(mau.mau, 0), 1) AS stickiness_pct
    FROM mau, dau
""")

# Ejecutamos SQL y recogemos el resultado en pandas.
with engine.connect() as conexion:
    resumen_actividad = pd.read_sql(consulta_mau, conexion)

resumen_actividad

,mau,dau_medio,stickiness_pct
0,7398,601.0,8.1


**Resumen — MAU y stickiness**

- **MAU:** jugadores distintos activos del 23/06 al 22/07/2024: **7.398**.
- **DAU medio:** **601**, calculado sobre todo el periodo del dataset.
- **Stickiness:** DAU medio / MAU × 100 = **8,1%** 


### Retención D1 — ¿vuelven al día siguiente?
- **Cohorte:** jugadores que ya pudieron cumplir su día 1 antes o en la
  fecha de extracción, el 22 de julio de 2024.
- **Retenidos:** jugadores de esa cohorte con una sesión en instalación + 1 día.
- **Retención D1:** retenidos / cohorte × 100.

Cada jugador cuenta una sola vez, aunque tenga varias sesiones ese día.

In [22]:
# Calculamos la retención D1
consulta_d1 = text("""
    WITH cohorte AS (
    -- Solo los que llegaron a D1
    SELECT user_id
    FROM public.usuarios
    WHERE DATE '2024-07-22' - fecha_instalacion >= 1
    ),

    retenidos AS (
    SELECT DISTINCT s.user_id 
    FROM public.sesiones AS s
    JOIN public.usuarios AS u
        ON s.user_id = u.user_id
    WHERE s.fecha - u.fecha_instalacion = 1
    )

    SELECT 
        COUNT(DISTINCT c.user_id) AS cohorte,
        COUNT(DISTINCT r.user_id) AS retenidos,
        -- Dividimos los retenidos entre los jugadores elegibles
        ROUND(
            100.0*COUNT(DISTINCT r.user_id)
            / NULLIF(COUNT(DISTINCT c.user_id),0),
            1
            ) AS retencion_d1_pct
    
    FROM cohorte AS c
    -- Conservamos también a quienes no volvieron para incluirlos en el total
    LEFT JOIN retenidos AS r
        ON c.user_id = r.user_id
""")

# Ejecutamos la consulta y guardamos su resultado en un DataFrame
with engine.connect() as conexion:
    retencion_d1 = pd.read_sql(consulta_d1,conexion)

retencion_d1

,cohorte,retenidos,retencion_d1_pct
0,8000,3261,40.8


**Resumen — retención D1**

- **Cohorte:** jugadores que pudieron cumplir D1 al corte del 22/07: **8.000**.
- **Retenidos:** tuvieron una sesión exactamente al día siguiente de instalar: **3.261**.
- **Fórmula:** retenidos / cohorte × 100 = **40,8%**.
- **Clave:** ser elegible no significa haber vuelto; varias sesiones en D1 cuentan una sola vez.


#### Retención D1, D7 y D30
¿Qué porcentaje de jugadores vuelve exactamente al día siguiente, a los siete días y a los treinta días de instalar?

Usamos el 22 de julio de 2024 como fecha de extracción. Para cada plazo calculamos:
- **Cohorte:** jugadores que ya tuvieron tiempo de alcanzar ese día.
- **Retenidos:** jugadores de esa cohorte con al menos una sesión
  exactamente en instalación + N días.
- **Retención:** retenidos / cohorte × 100.


In [33]:
# Calculamos la retención en los días 1, 7 y 30.
consulta_retencion = text("""
    -- Creamos los tres plazos que queremos evaluar.
    WITH dias(n) AS (VALUES (1), (7), (30))

    SELECT
        d.n AS dia,
        COUNT(DISTINCT u.user_id) AS cohorte,
        COUNT(DISTINCT s.user_id) AS retenidos,

        -- Porcentaje de jugadores elegibles que volvieron ese día.
        ROUND(
            100.0 * COUNT(DISTINCT s.user_id)
            / NULLIF(COUNT(DISTINCT u.user_id), 0),
            1
        ) AS retencion_pct

    FROM dias AS d

    -- Incluimos a quienes ya tuvieron tiempo de alcanzar cada plazo.
    JOIN public.usuarios AS u
        ON DATE '2024-07-22' - u.fecha_instalacion >= d.n

    -- Buscamos actividad exactamente N días DESPUÉS de instalar.
    LEFT JOIN public.sesiones AS s
        ON s.user_id = u.user_id
        AND s.fecha - u.fecha_instalacion = d.n

    GROUP BY d.n
    ORDER BY d.n
""")

# Ejecutamos la consulta corregida y actualizamos el DataFrame.
with engine.connect() as conexion:
    retencion = pd.read_sql(consulta_retencion, conexion)

retencion
retencion

,dia,cohorte,retenidos,retencion_pct
0,1,8000,3261,40.8
1,7,6808,1005,14.8
2,30,2470,176,7.1


**Resumen — retención D1 / D7 / D30**

- **Qué mide:** volver exactamente N días después de instalar; no en cualquier momento de los primeros N días.
- **D1 — primera vuelta:** aproximadamente 41 de cada 100 jugadores vuelven al día siguiente de instalar.
- **D7 — vuelta una semana después:** aproximadamente 15 de cada 100 jugadores elegibles juegan exactamente en su día 7.
- **D30 — vuelta treinta días después:** aproximadamente 7 de cada 100 elegibles juegan exactamente en su día 30.


#### Embudo de progresión - niveles 1,3,6,10
¿Cuántos jugadores llegan a jugar cada nivel clave?

Contamos jugadores únicos con al menos una partida registrada en los niveles 1, 3, 6 y 10, y calculamos su porcentaje sobre el total de usuarios.
- Cada jugador cuenta 1 vez por nivel
- Tener una partida en un nivel indica que lo jugó no que lo superó

In [26]:
# Contamos jugadores únicos en los niveles clave.
consulta_embudo = text("""
    SELECT 
        nivel,
        COUNT(DISTINCT user_id) AS jugadores,
        -- Dividimos entre el total de usuarios
        ROUND(
            100.0*COUNT(DISTINCT user_id)
            / NULLIF((SELECT COUNT(*) FROM public.usuarios),0),
            1
            ) AS pct_del_total
    FROM public.partidas
    WHERE nivel IN (1,3,6,10)
    GROUP BY nivel
    ORDER BY nivel
""")

# Ejecutamos la consulta y guardamos el resultado en pandas.
with engine.connect() as conexion:
    embudo = pd.read_sql(consulta_embudo, conexion)

embudo

,nivel,jugadores,pct_del_total
0,1,8000,100.0
1,3,6320,79.0
2,6,3680,46.0
3,10,960,12.0


**Resumen — embudo de niveles**

- **Qué cuenta:** jugadores con una partida en cada nivel exacto, sin repetir intentos del mismo jugador.
- **Resultados:** nivel 1: **8.000 (100%)**; nivel 3: **6.320 (79%)**; nivel 6: **3.680 (46%)**; nivel 10: **960 (12%)**.
- **Denominador:** los 8.000 usuarios, no los jugadores del nivel anterior.
- **Límite:** jugar un nivel no significa superarlo; la antigüedad de los jugadores no está ajustada.


#### El escalón entre los niveles 6 y 7
¿Qué porcentaje de quienes alcanzan el nivel 6 no llega al nivel 7?
La caída identifica un punto que investigar, pero no demuestra por sí sola un problema de diseño ni un abandono definitivo. 

In [35]:
# Contamos jugadores únicos en ambos niveles y calculamos la caída.
consulta_caida = text("""
    WITH niveles AS (
    SELECT 
        COUNT(
            DISTINCT CASE WHEN nivel >= 6 THEN user_id END
        ) AS llegan_al_6,
        COUNT(
            DISTINCT CASE WHEN nivel >= 7 THEN user_id END
        ) AS llegan_al_7
    FROM public.partidas

    )

    SELECT 
        llegan_al_6,
        llegan_al_7,
        ROUND(
        100.0 * (llegan_al_6 - llegan_al_7)
        / NULLIF(llegan_al_6,0),
        1
    ) AS caida_pct

    FROM niveles
""")

# Ejecutamos SQL y guardamos su respuesta en un DataFrame.
with engine.connect() as conexion:
    caida_nivel_6 = pd.read_sql(consulta_caida, conexion)

# Mostramos el resultado.
caida_nivel_6

,llegan_al_6,llegan_al_7,caida_pct
0,3680,1920,47.8


**Resumen — caída entre niveles 6 y 7**

- **Alcanzan al menos el nivel 6:** **3.680** jugadores (nivel >= 6).
- **Alcanzan al menos el nivel 7:** **1.920** jugadores (nivel >= 7).
- **Diferencia:** **1.760** jugadores; caída = (3.680 − 1.920) / 3.680 × 100 = **47,8%**.
- **Lectura:** punto para investigar intentos, dificultad y tiempo disponible; no prueba un problema de diseño ni abandono definitivo.


#### Sesiones por jugador y duración media
¿Con qué frecuencia abren el juego los jugadores activos y cuánto duran sus sesiones?
- **Sesiones por jugador:** total de sesiones / jugadores únicos presentes en la tabla de sesiones, durante todo el periodo.
- **Duración media:** promedio de duración de las sesiones entre 1 y 3,600 segundos, ambos incluidos.




In [36]:
# Calculamos sesiones por jugador activo y duración media de sesión.
consulta_sesiones = text("""
    SELECT 
        ROUND(
        COUNT(*)*1.0 / NULLIF(COUNT(DISTINCT user_id),0),
        2) AS sesiones_por_jugador,
    -- Duraciones entre 1 segundo y 1 hora
    ROUND(
        AVG(
            CASE        
                WHEN duracion_segundos BETWEEN 1 AND 3600
                THEN duracion_segundos
            END
        ),1   
    ) AS duracion_media_seg
    FROM public.sesiones
""")

# Ejecutamos la consulta y guardamos el resultado en pandas.
with engine.connect() as conexion:
    resumen_sesiones = pd.read_sql(consulta_sesiones, conexion)

resumen_sesiones

,sesiones_por_jugador,duracion_media_seg
0,5.57,183.6


**Resumen — sesiones y duración**

- **Sesiones por jugador activo:** **5,57** en todo el periodo; no son sesiones diarias.
- **Duración media por sesión:** **183,6 segundos**, aproximadamente **3,06 minutos**.
- **Filtro:** duraciones entre 1 y 3.600 segundos. La limpieza ya había excluido las sesiones fuera del rango.

#### Actividad por día de la semana
**Hipótesis:** los jugadores juegan más los fines de semana.
Contamos, para cada día de la semana:
- **Jugadores activos:** jugadores distintos con al menos una sesión
  en ese día de la semana durante todo el periodo.
- **Sesiones:** total de sesiones registradas.

In [38]:
consulta_semana = text("""
    SELECT
        EXTRACT(ISODOW FROM fecha)::INTEGER AS dia_num,
        -- Asignamos nombre a día
        CASE EXTRACT(ISODOW FROM fecha)
            WHEN 1 THEN 'lunes'
            WHEN 2 THEN 'martes'
            WHEN 3 THEN 'miercoles'
            WHEN 4 THEN 'jueves'
            WHEN 5 THEN 'viernes'
            WHEN 6 THEN 'sabado'
            WHEN 7 THEN 'domingo'
        END AS dia,
        COUNT(DISTINCT user_id) AS jugadores_Activos,
        COUNT(*) AS sesiones

    FROM public.sesiones
    GROUP BY dia_num, dia
    ORDER BY dia_num
""")


# Ejecutamos la consulta y mostramos el resultado.
with engine.connect() as conexion:
    actividad_semana = pd.read_sql(consulta_semana, conexion)

actividad_semana


,dia_num,dia,jugadores_activos,sesiones
0,1,lunes,3941,6695
1,2,martes,3564,5924
2,3,miercoles,3719,6127
3,4,jueves,3748,6316
4,5,viernes,3782,6333
5,6,sabado,3898,6404
6,7,domingo,3990,6774


**Resumen — actividad por día de la semana**

- **Jugadores activos:** personas distintas por día de la semana acumulado; quien juega varios lunes cuenta una vez en lunes. 
- **Sesiones:** todos los eventos de cada grupo, incluidos intentos repetidos del mismo jugador.
- **Resultados destacados:** domingo, **3.990 jugadores / 6.774 sesiones**; lunes, **3.941 / 6.695**; sábado, **3.898 / 6.404**.
- **Lectura:** domingo encabeza ambos recuentos, pero sábado no supera al lunes. No basta para afirmar que se juega mucho más todos los fines de semana ni para decidir un lanzamiento en sábado.
- **Límite:** estos agregados no demuestran el efecto que tendría lanzar un evento en un día concreto.


### Parte 2: Monetización — conversión, ARPU, ARPPU y ballenas

**Objetivo:** entender de dónde vienen los ingresos de PlayLoop y cómo se distribuyen entre los jugadores.

### Conversión a pagador
¿Qué porcentaje de jugadores ha realizado al menos una compra válida?
- **Jugadores**: usuarios registrados
- **Pagadores**: usuarios distintos presentes en la tabla de compras.
- **Conversión**: pagadores / jugaadores * 100

Utilizamos las compras no reembolsadas en euros conservadas durante la limpieza. El resultado corresponde al periodo observado completo.

In [41]:
# Calculamos el número de jugadores, pagadores únicos y su porcentaje.
consulta_conversion = text("""
    SELECT
        -- Subconsulta para el total de jugadores
        (SELECT COUNT(*) FROM public.usuarios) AS jugadores,
        -- Pagadores solo cuenta 1 vez
        COUNT(DISTINCT user_id) AS pagadores,
        -- Conversion
        ROUND(
        COUNT(DISTINCT user_id)*100.0/ NULLIF((SELECT COUNT(*) FROM public.usuarios),0),
        2)
    FROM public.compras
""")

# Enviamos la consulta a PostgreSQl y guardamos el resultado
with engine.connect() as conexion:
    conversion = pd.read_sql(consulta_conversion,conexion)

conversion

,jugadores,pagadores,round
0,8000,107,1.34


**Resumen — conversión a pagador**

- **Jugadores:** 8.000.
- **Pagadores únicos:** 107.
- **Conversión:** 1,34%; aproximadamente 1 de cada 75 jugadores compró.
- **Interpretación:** el ingreso por compras procede de una pequeña
  parte de la base de jugadores.
- **Alcance:** contamos compras no reembolsadas en euros durante el
  periodo observado. No comprar en este periodo no significa que
  el jugador nunca vaya a hacerlo.

#### Ingreso por compras: bruto, comisión y neto

¿Cuánto pagan los jugadores y cuánto queda para PlayLoop después de la comisión de la tienda?

- **Bruto:** suma de los importes de las compras válidas.
- **Comisión de tienda:** bruto × 0.30.
- **Neto de comisión:** bruto × 0.70.

Aplicamos la comisión del 30% establecida en el caso.

In [43]:
# Separamos el ingreso por compras y la comisión asumida en el caso.
consulta_ingresos_compras = text("""
    SELECT 
        ROUND(SUM(importe),2) AS bruto,
        ROUND(SUM(importe)*0.30,2) AS comision_tienda,
        ROUND(SUM(importe)*0.70, 2) AS neto

    FROM public.compras
""")

with engine.connect() as conexion:
    ingresos_compras = pd.read_sql(consulta_ingresos_compras,conexion)

ingresos_compras

,bruto,comision_tienda,neto
0,10838.71,3251.61,7587.1


**Resumen — bruto, comisión y neto**

- **Ingreso bruto por compras:** 10.838,71 €.
- **Comisión de tienda (30%):** 3.251,61 €.
- **Ingreso neto de comisión (70%):** 7.587,10 €.
- **Interpretación:** de cada euro pagado, PlayLoop conserva 0,70 € bajo el supuesto de comisión del caso.

#### ARPU — ingreso neto medio por jugador

¿Cuánto ingreso neto registrado genera, de media, cada jugador durante el periodo observado?

ARPU = (compras × 0.70 + ingresos publicitarios) / total de jugadores.

- Incluimos a todos los jugadores en el denominador, no solo a los pagadores.
- Aplicamos el 30% de comisión del caso únicamente a las compras.
- Los ingresos publicitarios se suman tal como están registrados.
- Es un promedio del periodo observado, no una estimación del valor
  completo del jugador durante toda su vida.

In [44]:
# Sumamos ambas fuentes de ingreso antes de repartir entre todos los usuarios.
consulta_arpu = text("""
    SELECT 
        ROUND(
            ((SELECT SUM(importe)*0.7 FROM public.compras) +
            (SELECT SUM(ingreso) FROM public.ingresos_ads))
            / NULLIF((SELECT COUNT(*) FROM public.usuarios), 0),
        3
        ) AS arpu_neto
""")

# Ejecutamos la consulta y guardamos su resultado en pandas.
with engine.connect() as conexion:
    arpu = pd.read_sql(consulta_arpu, conexion)

arpu

,arpu_neto
0,1.448


**Resumen — ARPU neto**
- **Resultado:** 1,448 € de ingreso neto registrado por jugador.
- **Interpretación:** repartimos el ingreso entre todos, paguen o no.

#### ARPPU bruto — gasto medio por pagador
¿Cuánto gastó, de media, cada jugador que realizó compras válidas?
Solo incluimos a quienes compran en el denominador. Los anuncios
no forman parte de este cálculo.

In [ ]:
# Calculamos el gasto bruto medio entre quienes realizaron compras válidas.
consulta_arppu = text("""
    SELECT 
        COUNT(DISTINCT user_id) AS pagadores,
        ROUND(SUM(importe),2) AS iap_bruto,
        ROUND(
        SUM(importe) / NULLIF(COUNT(DISTINCT user_id),0),
        1
        ) AS arppu

    FROM public.compras
""")

# Ejecutamos SQL y recogemos el resultado en pandas.
with engine.connect() as conexion:
    arppu = pd.read_sql(consulta_arppu,conexion)

arppu
                      

,pagadores,iap_bruto,arppu
0,107,10838.71,101.3


**Resumen — ARPPU bruto**
- **ARPPU bruto:** 101,3 € por pagador durante el periodo.
- **Interpretación:** cada comprador gastó esa cantidad de media, aunque el gasto puede estar muy concentrado en unos pocos.

#### Compras frente a anuncios — reparto del ingreso neto
¿Qué parte del ingreso neto registrado procede de cada vía?
- **IAP neto:** ingreso por compras × 0.70, descontando la comisión del caso.
- **Anuncios:** suma de los ingresos publicitarios registrados.
- **Peso del IAP:** IAP neto / (IAP neto + anuncios) × 100.

In [51]:
consulta_reparto = text("""
    SELECT 
        ROUND(
            (SELECT SUM(importe) * 0.7 FROM public.compras),
            2) AS iap_neto,
        ROUND(
            (SELECT SUM(ingreso) FROM public.ingresos_ads),
            2) AS ingresos_anuncios,
        ROUND(
            100.0 * (SELECT SUM(importe) * 0.70 FROM public.compras)
            / NULLIF(
                (SELECT SUM(importe) * 0.70 FROM public.compras)
                + (SELECT SUM(ingreso) FROM public.ingresos_ads),
                0
            ),
            1
        ) AS pct_iap
""")

# Ejecutamos y guardamos 
with engine.connect() as conexion:
    reparto_ingresos = pd.read_sql(consulta_reparto,conexion)

reparto_ingresos

,iap_neto,ingresos_anuncios,pct_iap
0,7587.1,3997.99,65.5


**Resumen — compras frente a anuncios**

- **Compras netas de comisión:** 7.587,10 € → 65,5% del total.
- **Ingresos publicitarios registrados:** 3.997,99 € → 34,5%.
- **Total registrado:** 11.585,09 €.
- **Interpretación:** las compras son la principal fuente de ingreso,
  aunque los anuncios aportan aproximadamente un tercio.

#### Concentración del ingreso — los mayores pagadores
¿Qué porcentaje del ingreso bruto por compras aporta el 2% de pagadores que más gasta?
Este cálculo permite evaluar cuánto depende el ingreso de unos pocos compradores. No incluye ingresos publicitarios.

In [48]:
# Calculamos qué porcentaje del ingreso por compras concentran
# los pagadores situados en el 2% superior del ranking.
consulta_ballenas = text("""
    WITH gasto_por_pagador AS (
        SELECT user_id, SUM(importe) AS gasto
        FROM compras 
        GROUP BY user_id
    ),
    ranking AS (
        SELECT 
            gasto,
            ROW_NUMBER() OVER (ORDER BY gasto DESC) AS rn,
            COUNT(*) OVER() AS total_pagadores
        FROM gasto_por_pagador
    )
    SELECT 
        CAST(CEIL(0.02*total_pagadores) AS INT) AS ballenas_top2pct,
        ROUND(
            100.0*SUM(CASE WHEN rn<= CEIL(0.02 * total_pagadores) THEN gasto END)
            / SUM(gasto)
        ,1) AS pct_ingresos_ballenas
    FROM ranking
    GROUP BY total_pagadores
""")

with engine.connect() as conexion:
    concentracion_ingresos = pd.read_sql(consulta_ballenas,conexion)

concentracion_ingresos


,ballenas_top2pct,pct_ingresos_ballenas
0,3,52.7


**Resumen — concentración del ingreso**

- **Grupo seleccionado:** 3 jugadores, el 2,8% de los 107 pagadores
  tras redondear hacia arriba el umbral del 2%.
- **Aportación:** generan el 52,7% del ingreso bruto por compras.
- **Interpretación:** más de la mitad del ingreso por compras depende
  de tres personas; perder su actividad podría afectar mucho al total.
- **Relación con el ARPPU:** estos grandes compradores elevan la media,
  por lo que conviene revisar también la mediana del gasto por pagador.

#### Qué se compra — ingreso por tipo de producto
¿Qué categorías de productos generan más ingreso por compras?

In [53]:
# Agrupamos las compras según el tipo de producto del catálogo.
consulta_productos = text("""
    SELECT 
        pi.tipo,
        COUNT(*) AS compras,
        ROUND(SUM(c.importe),2) AS ingreso_bruto

    FROM public.compras AS c
    JOIN public.productos_iap AS pi
        ON c.producto_id = pi.producto_id
    GROUP BY pi.tipo
    ORDER BY ingreso_bruto DESC
""")

# Ejecutamos SQL y guardamos el resultado en pandas.
with engine.connect() as conexion:
    compras_por_tipo = pd.read_sql(consulta_productos, conexion)

compras_por_tipo

,tipo,compras,ingreso_bruto
0,monedas,4300,10124.50
1,quitar_anuncios,179,714.21


**Resumen — compras por tipo de producto**

- **Monedas:** 4.300 compras y 10.124,50 € brutos, el 93,4% del ingreso.
- **Quitar anuncios:** 179 compras y 714,21 € brutos, el 6,6%.
- **Interpretación:** la monetización por compras depende principalmente
  de los productos clasificados como monedas.

#### Conversión a pagador por plataforma
¿Qué porcentaje de los usuarios de iOS y Android realizó al menos una compra válida?
- **Jugadores:** usuarios registrados en cada plataforma.
- **Pagadores:** usuarios de esa plataforma con alguna compra válida.
- **Conversión:** pagadores / jugadores de la plataforma × 100.

Utilizamos la plataforma registrada en usuarios y contamos personas distintas, no transacciones. Una mayor conversión no implica necesariamente un mayor gasto por comprador.

In [56]:
# Calculamos la conversión por plataforma usando una lista de pagadores únicos.
consulta_conversion_plataforma = text("""
    WITH pagadores AS(
        SELECT DISTINCT user_id FROM public.compras)

    SELECT 
        u.plataforma,
        COUNT(DISTINCT u.user_id) AS jugadores,
        COUNT(DISTINCT p.user_id) AS pagadores,
        ROUND(
            100.0*COUNT(DISTINCT p.user_id) 
            / NULLIF(COUNT(DISTINCT u.user_id),0),
            2
        ) AS conversion_pct

    FROM public.usuarios AS u
    LEFT JOIN pagadores AS p
        ON u.user_id = p.user_id
    GROUP BY u.plataforma
    ORDER BY conversion_pct DESC
""")

with engine.connect() as conexion:
    conversion_plataforma =  pd.read_sql(consulta_conversion_plataforma,conexion)

conversion_plataforma

,plataforma,jugadores,pagadores,conversion_pct
0,iOS,3646,57,1.56
1,Android,4354,50,1.15


**Resumen — conversión por plataforma**
- **iOS:** 57 pagadores entre 3.646 jugadores → 1,56%.
- **Android:** 50 pagadores entre 4.354 jugadores → 1,15%.
- **Interpretación:** iOS tiene menos jugadores, pero una mayor proporción
  realizó compras válidas durante el periodo.
- **Diferencia:** 0,41 puntos porcentuales a favor de iOS.

#### Media frente a mediana — ¿cuánto gasta el pagador habitual?
Primero sumamos las compras de cada pagador y después comparamos:

- **Media o ARPPU bruto:** gasto total dividido entre pagadores.
- **Mediana:** gasto situado en el centro al ordenar a los pagadores.
- **Máximo:** mayor gasto acumulado de un pagador.

La media es sensible a los grandes compradores. La mediana ayuda
a describir el centro del reparto sin que esos extremos la arrastren.

Utilizamos compras brutas válidas durante el periodo observado.

In [58]:
# Comparamos el gasto acumulado de cada pagador.
consulta_gasto_pagador = text("""
    WITH gasto_por_pagador AS (
        SELECT user_id, SUM(importe) AS gasto
        FROM compras
        GROUP BY user_id
    )
    SELECT 
        ROUND(AVG(gasto),2) AS arppu_media,
        ROUND(
            CAST(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY gasto)
                AS NUMERIC
                ),
            2) AS gasto_mediano,
        ROUND(MAX(gasto),2) AS gasto_max

    FROM gasto_por_pagador
        
""")

# Ejecutamos la consulta y guardamos el resultado en pandas.
with engine.connect() as conexion:
    gasto_pagadores = pd.read_sql(consulta_gasto_pagador, conexion)

gasto_pagadores

,arppu_media,gasto_mediano,gasto_max
0,101.3,29.4,2658.53


**Resumen — media y mediana del gasto por pagador**

- **Media:** 101,3 € por pagador.
- **Mediana:** 29,4 €; 53 pagadores gastaron esa cantidad o menos
  y otros 53 esa cantidad o más.
- **Máximo:** un solo pagador acumuló 2.658,53 €.
- **Interpretación:** el gasto está concentrado; la media no describe
  bien a todos los compradores.
- **Decisión:** no podemos garantizar 100.000 € por captar 1.000 pagadores.
